In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# 0. Load merged firm + macro datasets (unscaled & scaled)
# ============================================================

path_merged = "../data/final/firm_macro_merged.csv"
path_merged_scaled = "../data/final/firm_macro_merged_scaled.csv"

df = pd.read_csv(path_merged)
df_scaled = pd.read_csv(path_merged_scaled)

print("Unscaled merged shape:", df.shape)
print("Scaled merged shape:", df_scaled.shape)

print("\nHead of unscaled merged data:")
display(df.head())

# ============================================================
# 1. Type casting & basic checks
# ============================================================

# Ensure key columns exist
required_cols = ["gvkey", "fyear", "bankruptcy_dlrsn"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in merged data: {missing}")

# Cast types
df["gvkey"] = df["gvkey"].astype(str)
df_scaled["gvkey"] = df_scaled["gvkey"].astype(str)

df["fyear"] = pd.to_numeric(df["fyear"], errors="coerce").astype("Int64")
df_scaled["fyear"] = pd.to_numeric(df_scaled["fyear"], errors="coerce").astype("Int64")

df["bankruptcy_dlrsn"] = df["bankruptcy_dlrsn"].astype(int)

print("\nBankruptcy label distribution (bankruptcy_dlrsn):")
print(df["bankruptcy_dlrsn"].value_counts(dropna=False))

# Tri par firme / année
df = df.sort_values(["gvkey", "fyear"]).reset_index(drop=True)

# ============================================================
# 2. Compute event year per firm (first bankruptcy year)
# ============================================================

# event_year per gvkey = première année où bankruptcy_dlrsn == 1
event_year_series = (
    df.loc[df["bankruptcy_dlrsn"] == 1]
      .groupby("gvkey")["fyear"]
      .min()
)

# Map to all rows
df["event_year"] = df["gvkey"].map(event_year_series)

print("\nExamples of event_year (first 10 non-null):")
display(df.loc[df["event_year"].notna(), ["gvkey", "fyear", "event_year"]].head(10))

# ============================================================
# 3. Years to event (only for pre-event years)
# ============================================================

# years_to_event = event_year - fyear
df["years_to_event"] = df["event_year"] - df["fyear"]

# Pour les firmes sans faillite, event_year est NaN → years_to_event NaN
# Pour les années après ou à l'année de faillite, years_to_event <= 0

print("\nDistribution of years_to_event (non-null):")
display(df["years_to_event"].dropna().describe())

# ============================================================
# 4. Create distress horizons: 1y, 2y, 3y
#    - 1 si faillite dans les H ans SUIVANTS (1 ≤ years_to_event ≤ H)
#    - 0 sinon
# ============================================================

horizons = [1, 2, 3]

for h in horizons:
    col = f"distress_{h}y"
    df[col] = np.where(
        df["years_to_event"].between(1, h, inclusive="both").fillna(False),
        1,
        0
    )

print("\nValue counts for distress horizons:")
for h in horizons:
    col = f"distress_{h}y"
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

# ============================================================
# 5. Optional: remove post-event years from modeling sample
#    (années >= event_year)
#    Ici on garde tout dans le fichier final mais on crée un flag
# ============================================================

df["post_event_or_event_year"] = (
    df["event_year"].notna() & (df["fyear"] >= df["event_year"])
).astype(int)

print("\npost_event_or_event_year flag distribution:")
print(df["post_event_or_event_year"].value_counts())

# Pour ton modeling, tu pourras filtrer:
# df_model = df[df["post_event_or_event_year"] == 0].copy()

# ============================================================
# 6. Join labels back to SCALED dataset
# ============================================================

label_cols = ["event_year", "years_to_event", "post_event_or_event_year"] + [
    f"distress_{h}y" for h in horizons
]

df_labels = df[["gvkey", "fyear"] + label_cols].copy()

df_scaled["gvkey"] = df_scaled["gvkey"].astype(str)
df_scaled["fyear"] = pd.to_numeric(df_scaled["fyear"], errors="coerce").astype("Int64")

df_scaled_with_labels = df_scaled.merge(
    df_labels,
    on=["gvkey", "fyear"],
    how="left"
)

print("\nScaled + labels shape:", df_scaled_with_labels.shape)
display(df_scaled_with_labels.head())

# ============================================================
# 7. Save final datasets with distress horizons
# ============================================================

output_unscaled = "../data/final/firm_macro_with_distress.csv"
output_scaled = "../data/final/firm_macro_with_distress_scaled.csv"

df.to_csv(output_unscaled, index=False)
df_scaled_with_labels.to_csv(output_scaled, index=False)

print("\nSaved unscaled dataset with distress labels to:", output_unscaled)
print("Saved SCALED dataset with distress labels to:", output_scaled)

# Petit résumé
print("\nSummary:")
print("  Unscaled merged + labels:", df.shape)
print("  Scaled merged + labels:", df_scaled_with_labels.shape)


Unscaled merged shape: (240308, 29)
Scaled merged shape: (240308, 29)

Head of unscaled merged data:


,gvkey,datadate,fyear,conm,tic,dlrsn,bankruptcy_dlrsn,roa,ebitda_margin,debt_ratio,...,book_to_market,price_to_book,working_capital_to_assets,retained_earnings_to_assets,name,us_gdp_(ar)_cura,us_cpi_-_all_urban:_all_items_sadj,us_treasury_bill_rate_-_3_month_(ep)_nadj,us_treasury_yield_adjusted_to_constant_maturity_-_20_year_nadj,us_unemployment_rate_sadj
0,1004,1995-05-31,1994,AAR CORP,AIR,NaN,0,0.031793,0.077019,0.537077,...,0.460255,1.234818,0.583569,0.240565,1994.31,7396.51572,149.51398,5.70,7.3257,6.1
1,1004,1996-05-31,1995,AAR CORP,AIR,NaN,0,0.049849,0.084273,0.532632,...,0.460255,1.729691,0.590680,0.250182,1995.00,7639.74900,152.38300,5.08,6.9600,5.6
2,1004,1997-05-31,1996,AAR CORP,AIR,NaN,0,0.060621,0.093627,0.491565,...,0.460255,2.095840,0.593143,0.231646,1996.00,8073.12200,156.85800,5.15,6.8200,5.4
3,1004,1998-05-31,1997,AAR CORP,AIR,NaN,0,0.074896,0.101006,0.551344,...,0.460255,2.434527,0.476098,0.220100,1997.00,8577.55300,160.52500,5.35,6.6800,4.9
4,1004,1999-05-31,1998,AAR CORP,AIR,NaN,0,0.080941,0.102876,0.551305,...,0.602903,1.658646,0.460482,0.245524,1998.00,9062.81700,163.00800,4.47,5.7200,4.5



Bankruptcy label distribution (bankruptcy_dlrsn):
bankruptcy_dlrsn
0    234314
1      5994
Name: count, dtype: int64

Examples of event_year (first 10 non-null):


,gvkey,fyear,event_year
85,10005,1995,1995
86,10005,1996,1995
87,10005,1997,1995
88,10005,1998,1995
89,10005,1999,1995
90,10005,2000,1995
91,10005,2001,1995
92,10005,2002,1995
93,10005,2003,1995
94,10005,2004,1995



Distribution of years_to_event (non-null):


count      5994.0
mean    -6.041208
std      5.674302
min         -28.0
25%          -9.0
50%          -4.0
75%          -2.0
max           0.0
Name: years_to_event, dtype: Float64


Value counts for distress horizons:

distress_1y:
distress_1y
0    240308
Name: count, dtype: int64

distress_2y:
distress_2y
0    240308
Name: count, dtype: int64

distress_3y:
distress_3y
0    240308
Name: count, dtype: int64

post_event_or_event_year flag distribution:
post_event_or_event_year
0    234314
1      5994
Name: count, dtype: int64

Scaled + labels shape: (242802, 35)


,gvkey,datadate,fyear,conm,tic,dlrsn,bankruptcy_dlrsn,roa,ebitda_margin,debt_ratio,...,us_cpi_-_all_urban:_all_items_sadj,us_treasury_bill_rate_-_3_month_(ep)_nadj,us_treasury_yield_adjusted_to_constant_maturity_-_20_year_nadj,us_unemployment_rate_sadj,event_year,years_to_event,post_event_or_event_year,distress_1y,distress_2y,distress_3y
0,1004,1995-05-31,1994,AAR CORP,AIR,NaN,0,0.226063,0.171086,-0.181664,...,-1.499747,1.514148,1.871440,0.317004,<NA>,<NA>,0,0,0,0
1,1004,1996-05-31,1995,AAR CORP,AIR,NaN,0,0.238559,0.171952,-0.183780,...,-1.434948,1.229085,1.641494,0.014970,<NA>,<NA>,0,0,0,0
2,1004,1997-05-31,1996,AAR CORP,AIR,NaN,0,0.246015,0.173069,-0.203332,...,-1.333876,1.261269,1.553465,-0.105844,<NA>,<NA>,0,0,0,0
3,1004,1998-05-31,1997,AAR CORP,AIR,NaN,0,0.255895,0.173950,-0.174871,...,-1.251054,1.353225,1.465435,-0.407879,<NA>,<NA>,0,0,0,0
4,1004,1999-05-31,1998,AAR CORP,AIR,NaN,0,0.260078,0.174173,-0.174890,...,-1.194974,0.948620,0.861805,-0.649506,<NA>,<NA>,0,0,0,0



Saved unscaled dataset with distress labels to: ../data/final/firm_macro_with_distress.csv
Saved SCALED dataset with distress labels to: ../data/final/firm_macro_with_distress_scaled.csv

Summary:
  Unscaled merged + labels: (240308, 35)
  Scaled merged + labels: (242802, 35)
